In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


In [ ]:
df = pd.read_csv(r'../data/students_mental_health_survey.csv')
print(df.head())


In [ ]:
df.info()
print(df.describe())
print('\nMissing Values:')
print(df.isnull().sum())


In [ ]:
plt.figure(figsize=(6,4))
df['Depression_Score'].hist(bins=30)
plt.title('Distribution of Depression Scores')
plt.xlabel('Depression Score')
plt.ylabel('Frequency')
plt.show()


In [ ]:
print('Sample sizes by Course:')
print(df['Course'].value_counts())

print('\nSample sizes by Gender:')
print(df['Gender'].value_counts())

print('\nSample sizes by Age:')
print(df['Age'].value_counts().sort_index())


In [ ]:
course_stats = df.groupby('Course')['Depression_Score'].agg(['mean', 'std', 'count']).sort_values('mean')
print('Depression Stats by Course:')
print(course_stats)

plt.figure(figsize=(10,5))
order = course_stats.index.tolist()
sns.boxplot(data=df, x='Course', y='Depression_Score', order=order)
plt.title('Depression Score Distribution by Course')
plt.xlabel('Course')
plt.ylabel('Depression Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
gender_stats = df.groupby('Gender')['Depression_Score'].agg(['mean', 'std', 'count'])
print('Depression Stats by Gender:')
print(gender_stats)

plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='Gender', y='Depression_Score')
plt.title('Depression Score Distribution by Gender')
plt.ylabel('Depression Score')
plt.show()


In [ ]:
age_stats = df.groupby('Age')['Depression_Score'].agg(['mean', 'std', 'count'])
print('Depression Stats by Age:')
print(age_stats)

plt.figure(figsize=(8,4))
sns.boxplot(data=df, x='Age', y='Depression_Score')
plt.title('Depression Score Distribution by Age')
plt.xlabel('Age')
plt.ylabel('Depression Score')
plt.show()


In [ ]:
interaction = df.groupby(['Gender', 'Course'])['Depression_Score'].mean()
print('Depression by Gender and Course:')
print(interaction)


In [ ]:
course_mean = df.groupby('Course')['Depression_Score'].mean().sort_values()

print('\nHighest Depression Courses:')
print(course_mean.tail(3))

print('\nLowest Depression Courses:')
print(course_mean.head(3))


In [ ]:
print('Overall Depression Stats:')
print(df['Depression_Score'].describe())


In [ ]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

outliers_count = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outliers_count[col] = len(outliers)

print('Outlier counts per column:')
print(outliers_count)

# Decision: Age has 131 outliers (ages 30-35), but these are real, valid data
# points representing mature students. Removing them would bias the dataset.
# CGPA has 16 outliers — also kept as they are plausible academic scores.
# All outliers are RETAINED. No clipping or removal applied.
print('\nDecision: All outliers retained — values are real and domain-valid.')


In [ ]:
# Encode first so categorical variables (e.g. Course) appear in correlation
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print('Encoded dataframe shape:', df_encoded.shape)

corr = df_encoded.corr(numeric_only=True)['Depression_Score'].sort_values(ascending=False)
print('\nCorrelation with Depression Score (encoded):')
print(corr)


In [ ]:
X = df_encoded.drop('Depression_Score', axis=1)
y = df_encoded['Depression_Score']

# One global split used by ALL models (multi- and single-factor)
# This ensures consistent, comparable evaluation across every model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]} rows | Test size: {X_test.shape[0]} rows')


In [ ]:
# Impute missing values using train statistics only (prevents data leakage)
# The imputer learns the mean/mode from training data, then applies it to test
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
bool_features = X_train.select_dtypes(include=['bool']).columns.tolist()

num_imputer = SimpleImputer(strategy='mean')
X_train_num = X_train[numeric_features].copy()
X_test_num = X_test[numeric_features].copy()

X_train[numeric_features] = num_imputer.fit_transform(X_train_num)
X_test[numeric_features] = num_imputer.transform(X_test_num)  # transform only — no fit

print('Imputation complete. Any remaining nulls in train:')
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
print('Any remaining nulls in test:')
print(X_test.isnull().sum()[X_test.isnull().sum() > 0])


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print('Linear Regression:')
print(f'  R²: {r2_score(y_test, y_pred_lr):.4f}')
print(f'  RMSE: {root_mean_squared_error(y_test, y_pred_lr):.4f}')


In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
print('Ridge Regression:')
print(f'  R²: {r2_score(y_test, y_pred_ridge):.4f}')
print(f'  RMSE: {root_mean_squared_error(y_test, y_pred_ridge):.4f}')


In [ ]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print('Lasso Regression:')
print(f'  R²: {r2_score(y_test, y_pred_lasso):.4f}')
print(f'  RMSE: {root_mean_squared_error(y_test, y_pred_lasso):.4f}')


In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('Random Forest Regressor:')
print(f'  R²: {r2_score(y_test, y_pred_rf):.4f}')
print(f'  RMSE: {root_mean_squared_error(y_test, y_pred_rf):.4f}')


In [ ]:
feature_importance = pd.Series(rf.feature_importances_, index=X.columns)
feature_importance.sort_values(ascending=False).head(10).plot(kind='bar')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()


In [ ]:
# Single-factor models using the same global train/test split
# This makes R² values directly comparable to the multi-factor models
single_factors = ['Stress_Level', 'Anxiety_Score', 'Financial_Stress', 'CGPA', 'Age']

print('Single-Factor R² Scores (Linear Regression):')
for factor in single_factors:
    m = LinearRegression()
    m.fit(X_train[[factor]], y_train)
    r2 = r2_score(y_test, m.predict(X_test[[factor]]))
    print(f'  {factor}: R² = {r2:.4f}')


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Helper: build a single-row input matching the encoded feature space ───────
def build_input_row(course, gender, age, cgpa, stress, anxiety, financial):
    """
    Recreates the exact one-hot encoding that pd.get_dummies(drop_first=True)
    produced during training so the model receives the right feature vector.
    """
    # Base numeric features
    row = {
        'Age':                age,
        'CGPA':               cgpa,
        'Stress_Level':       stress,
        'Anxiety_Score':      anxiety,
        'Financial_Stress':   financial,
    }

    # --- Course dummies (reference = Business, so Business cols absent) ---
    for c in ['Computer Science', 'Engineering', 'Law', 'Medical', 'Others']:
        row[f'Course_{c}'] = int(course == c)

    # --- Gender dummy (reference = Female) ---
    row['Gender_Male'] = int(gender == 'Male')

    # Fill every other column that exists in X_train with 0
    # (covers all other one-hot cols for Sleep_Quality, Physical_Activity, etc.)
    full_row = pd.DataFrame([row])
    full_row = full_row.reindex(columns=X_train.columns, fill_value=0)
    return full_row

# ── Score band helper ─────────────────────────────────────────────────────────
def score_band(score):
    if score <= 1.5:
        return "Low", "#2ecc71"
    elif score <= 3.0:
        return "Moderate", "#f39c12"
    else:
        return "High", "#e74c3c"

# ── Widgets ───────────────────────────────────────────────────────────────────
style   = {'description_width': '160px'}
layout  = widgets.Layout(width='420px')

w_course    = widgets.Dropdown(
    options=['Business','Computer Science','Engineering','Law','Medical','Others'],
    description='Course:', style=style, layout=layout)

w_gender    = widgets.Dropdown(
    options=['Female','Male'],
    description='Gender:', style=style, layout=layout)

w_age       = widgets.BoundedIntText(
    value=20, min=18, max=35,
    description='Age:', style=style, layout=layout)

w_cgpa      = widgets.BoundedFloatText(
    value=3.0, min=0.0, max=4.0, step=0.01,
    description='CGPA (0–4):', style=style, layout=layout)

w_stress    = widgets.IntSlider(
    value=2, min=0, max=5,
    description='Stress Level:', style=style, layout=layout)

w_anxiety   = widgets.IntSlider(
    value=2, min=0, max=5,
    description='Anxiety Score:', style=style, layout=layout)

w_financial = widgets.IntSlider(
    value=2, min=0, max=5,
    description='Financial Stress:', style=style, layout=layout)

btn_predict = widgets.Button(
    description='Predict My Score',
    button_style='primary',
    layout=widgets.Layout(width='200px', margin='12px 0 0 0'))

out = widgets.Output()

# ── Prediction callback ───────────────────────────────────────────────────────
def on_predict(b):
    with out:
        clear_output()
        row = build_input_row(
            course    = w_course.value,
            gender    = w_gender.value,
            age       = w_age.value,
            cgpa      = w_cgpa.value,
            stress    = w_stress.value,
            anxiety   = w_anxiety.value,
            financial = w_financial.value,
        )

        pred = float(lr.predict(row)[0])
        pred = round(max(0, min(5, pred)), 2)   # clamp to [0, 5]
        band, colour = score_band(pred)
        dataset_mean = 2.25

        # Context line
        if w_course.value == 'Computer Science':
            context = ("Computer Science students show the highest average "
                       "depression score in this dataset (3.30 vs overall mean 2.25).")
        elif pred < dataset_mean:
            context = f"Your predicted score is below the dataset average of {dataset_mean}."
        else:
            context = f"Your predicted score is above the dataset average of {dataset_mean}."

        print("=" * 52)
        print("       STUDENT MENTAL HEALTH — PREDICTION")
        print("=" * 52)
        print(f"  Predicted Depression Score : {pred:.2f} / 5.00")
        print(f"  Risk Band                  : {band}")
        print(f"  Dataset Average            : {dataset_mean}")
        print("-" * 52)
        print(f"  {context}")
        print("=" * 52)
        print()
        print("  Bands:  Low 0.0–1.5  |  Moderate 1.5–3.0  |  High 3.0–5.0")
        print()
        print("  ⚠  DISCLAIMER: This prediction is generated by a")
        print("  machine-learning model trained on survey data.")
        print("  It is NOT a clinical diagnosis. If you are")
        print("  struggling, please speak to a professional.")
        print("=" * 52)

btn_predict.on_click(on_predict)

# ── Layout & display ──────────────────────────────────────────────────────────
title = widgets.HTML("<h3 style='margin-bottom:8px'>🎓 Student Depression Score Predictor</h3>")
form  = widgets.VBox([
    title,
    w_course, w_gender, w_age, w_cgpa,
    w_stress, w_anxiety, w_financial,
    btn_predict,
    out
])
display(form)
